In [ ]:
# Log sequence recorded on 2025/03/13 20:53:13
import time
import zhinst.core
import matplotlib.pyplot as plt
import numpy as np
from DeviceClasses.E5063A_VNA_Class_new import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB, dBm_to_watts, watts_to_dBm, RMS_to_dBm, dBm_to_RMS
from datetime import datetime
from DeviceClasses.Utilities.data_management import save_data_3
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
VNA_address = "TCPIP::134.219.128.52::INSTR"
SG_address  = 'TCPIP::134.219.128.14::INSTR'
FSW_address = "TCPIP::134.219.128.35::hislip0"

# #plots:
# fig_SA, ax_SA = plt.subplots()
# ax_SA.set(title = "Spectrum", xlabel = "frequency[Hz]", ylabel = "power-drive_power[dBm]")
   
#general parameters:

## VNA parameters
channel = 1     # formatting: [1,2,3,4]
measuring = ["S21","S12","S11","S22"] # formatting: ["S12", "S21"..]
traces = ["1"] # formatting: ["CH1TR1", "CH2TR1"]
span=0

number_of_point=10001
band_width = 1
power=0

#number of S21 sweep

##SG parameters

SG_freq = 7e9

freqs = np.concatenate((np.linspace(0, 1e3, 100),np.linspace(0, 100e3, int(1e3))))


base_path = r"C:\Experiments\Bulk_He_2\Attenuation\circuit_2_1\data\2025_03_13_circuit_2_1_B1_to_Zurich\\"
raw_data_path = base_path + r"raw_data\\"



In [ ]:

time_now = datetime.now()
timestamp_S21 = time_now.strftime("%Y_%m_%d_%H_%M_%S_%f")
                    
for probe_power in [-10,-5,0]:
    for SG_power in [10, -20, -10, 0, 5]:
        voltage = []
        voltage_sig = []

        for rel_freq in freqs:
            freq_start = rel_freq+SG_freq-0/2
            freq_stop = rel_freq+SG_freq+0/2

            with Signal_Generator(SG_address) as SG:
                with VNA(VNA_address, ) as vna:
                    SG.set_mw_frequency(SG_freq)
                    SG.set_mw_power(SG_power)
                    SG.turn_mw("On")

                    vna.sweep(1, "S21", "1", freq_start, freq_stop, number_of_point,\
                                            probe_power, band_width, calibration= "OFF",\
                                            _file_name="delete", file_location= base_path, reset = True, dummy_sweep = True)

                    daq = zhinst.core.ziDAQServer('127.0.0.1', 8005, 1)

                    daq_module = daq.dataAcquisitionModule()
                    daq_module.set('triggernode', '/dev1374/demods/0/sample.R')
                    daq_module.set('preview', 1)
                    daq_module.set('device', 'dev1374')
                    daq_module.set('historylength', 100)
                    daq_module.set('bandwidth', 0)
                    daq_module.set('hysteresis', 0.01)
                    daq_module.set('level', 0.1)
                    daq_module.set('save/directory', 'C:\\Users\\Data Analysis\\Documents\\Zurich Instruments\\LabOne\\WebServer')
                    daq_module.set('clearhistory', 1)
                    daq_module.set('clearhistory', 1)
                    daq_module.set('bandwidth', 0)
                    daq.setDouble('/dev1374/oscs/0/freq', rel_freq)
                    daq.setDouble('/dev1374/demods/0/timeconstant', 3.46145642e-05)
                    daq.setDouble('/dev1374/demods/0/rate', 5000)
                    daq_module.set('type', 0)
                    daq_module.set('clearhistory', 1)
                    daq_module.set('bandwidth', 0)
                    daq_module.set('endless', 0)
                    daq_module.subscribe('/dev1374/demods/0/sample.R.avg')
                    daq_module.execute()
                    # To read the acquired data from the module, use a
                    # while loop like the one below. This will allow the
                    # data to be plotted while the measurement is ongoing.
                    # Note that any device nodes that enable the streaming
                    # of data to be acquired, must be set before the while loop.
                    result = 0
                    while daq_module.progress() < 1.0 and not daq_module.finished():
                        time.sleep(1)
                        result = daq_module.read()
                        print(f"Progress {float(daq_module.progress()) * 100:.2f} %\r")
                    daq_module.finish()
                    daq_module.unsubscribe('*')

                    voltage_raw = np.abs(np.array(result["dev1374"]["demods"]["0"]["sample.r.avg"][0]["value"][0], dtype = float))
                    timestamps = np.array(result["dev1374"]["demods"]["0"]["sample.r.avg"][0]["timestamp"][0], dtype = float)
                    print(voltage_raw)
                    print(voltage_raw.dtype)
                    print(f"mean:{np.mean(voltage_raw)}")
                    
                    print(voltage_raw.size)
                    print(timestamps.size)

                    meta_data_raw = dict(result)
                    data_raw = np.array([timestamps, voltage_raw])
                    headers_raw = "timestamps[unknown]\tvoltage[V]"

                    file_name_raw = f"probepower_{probe_power}_SGpower_{SG_power}_rel_freq_{rel_freq}Hz.tsv"

                    save_data_3(data_raw, file_name_raw, raw_data_path, headers_raw, meta_data = meta_data_raw)

                    voltage.append(np.mean(voltage_raw))
                    voltage_sig.append(np.std(voltage_raw))

                    SG.turn_mw("Off")

        meta_data = {
            "vna span":span,
            "vna number_of_point":number_of_point,
            "vna channel":1,
            "zurich_tc":3.46145642e-05,
            "probe_power":power,
            "SG_freq":SG_freq,
            "VNA_cal":"OFF",
        }

        data = np.array([freqs, voltage, voltage_sig])
        headers = "frequency[Hz]\tVoltage[V]"



        file_name = f"{timestamp_S21}_zurichatten_voltage_SGpower_{SG_power}dbm_probepower_{probe_power}dbm.tsv"

        save_data_3(data, file_name, base_path, headers, meta_data)

            

In [ ]:
print(result["dev1374"]["demods"]["0"]["sample.r.avg"][0].keys())

In [ ]:
print(result["dev1374"]["demods"]["0"]["sample.r.avg"][0]["timestamp"][0])

In [ ]:
try:
    plt.close()
except Exception:
    pass

rel_time = timestamps- timestamps[0]


fig, ax = plt.subplots()
ax.set(xlabel = "time[unknown]", ylabel = "Amplitude[mV]")
plt.plot(rel_time ,voltage*1000)
